# 01. Análisis Exploratorio y Preparación de Datos

**Solventa - Prueba Técnica Data Scientist Jr.**

Exploración de datos, feature engineering y preprocesamiento.

Parte 1: Desarrollo de Modelo Predictivo
Solventa - Prueba Técnica Data Scientist Jr.

Objetivo: Construir un modelo que califique clientes y seleccione aquellos
con menor probabilidad de incumplimiento para un nuevo producto de alto riesgo.

Variable objetivo: Mora30 (justificación en el análisis)

In [1]:
import warnings

In [2]:
warnings.filterwarnings("ignore")

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    classification_report,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler
import pickle
import os

In [4]:
# Create output directories
os.makedirs("../output/figures", exist_ok=True)

1. CARGA DE DATOS

In [5]:
print("=" * 60)
print("PARTE 1: DESARROLLO DE MODELO PREDICTIVO")
print("=" * 60)

print("\n[1] Cargando datos...")
df = pd.read_excel("../ProductoNuevo.xlsx")
print(f"   Registros: {df.shape[0]}")
print(f"   Variables: {df.shape[1]}")

PARTE 1: DESARROLLO DE MODELO PREDICTIVO

[1] Cargando datos...


   Registros: 4315
   Variables: 22


2. SELECCIÓN DE VARIABLE OBJETIVO

In [6]:
print("\n[2] Selección de variable objetivo...")
print(f"\n   Mora30: {df['Mora30'].mean() * 100:.2f}% positivos")
print(f"   Mora60: {df['Mora60'].mean() * 100:.2f}% positivos")
print("""
   DECISIÓN: Se utiliza MORA30 como variable objetivo.
   Justificación:
   - Para productos de alto riesgo, detectar morosidad temprana (30 días)
     es más prudente que esperar 60 días.
   - Permite acciones de cobro preventivas más rápidas.
   - Mayor proporción de eventos positivos (24.4%) mejora la capacidad
     del modelo para aprender patrones de riesgo.
   - Mora30 captura un espectro más amplio de clientes problemáticos,
     reduciendo el riesgo de originación.""")

target = "Mora30"


[2] Selección de variable objetivo...

   Mora30: 24.36% positivos
   Mora60: 11.68% positivos

   DECISIÓN: Se utiliza MORA30 como variable objetivo.
   Justificación:
   - Para productos de alto riesgo, detectar morosidad temprana (30 días)
     es más prudente que esperar 60 días.
   - Permite acciones de cobro preventivas más rápidas.
   - Mayor proporción de eventos positivos (24.4%) mejora la capacidad
     del modelo para aprender patrones de riesgo.
   - Mora30 captura un espectro más amplio de clientes problemáticos,
     reduciendo el riesgo de originación.


3. ANÁLISIS EXPLORATORIO (EDA)

In [7]:
print("\n[3] Realizando análisis exploratorio...")

# 3a. Distribución de variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    ["No Mora", "Mora 30+"],
    [df[target].value_counts()[0], df[target].value_counts()[1]],
    color=["#2ecc71", "#e74c3c"],
)
axes[0].set_title(f"Distribución de {target}")
axes[0].set_ylabel("Cantidad de clientes")
for i, v in enumerate([df[target].value_counts()[0], df[target].value_counts()[1]]):
    axes[0].text(
        i, v + 50, f"{v} ({v / len(df) * 100:.1f}%)", ha="center", fontweight="bold"
    )

# 3b. Correlaciones numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(
    ["ID", "FFECHA", target]
)
corr = df[numeric_cols.tolist() + [target]].corr()[target].sort_values(ascending=False)

colors = ["#e74c3c" if c > 0 else "#3498db" for c in corr.drop(target)]
axes[1].barh(range(len(corr) - 1), corr.drop(target).values, color=colors)
axes[1].set_yticks(range(len(corr) - 1))
axes[1].set_yticklabels(corr.drop(target).index, fontsize=8)
axes[1].set_title("Correlación con Mora30")
axes[1].axvline(x=0, color="black", linewidth=0.5)

plt.tight_layout()
plt.savefig("../output/figures/01_eda_overview.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 01_eda_overview.png")

# 3c. Análisis de variables categóricas
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
categorical_cols = [
    "OCUPACION",
    "TIPCONTRATO",
    "Estado_Civil",
    "Genero",
    "Nivel_Academico",
    "Tipo_Vivienda",
]

for ax, col in zip(axes.flatten(), categorical_cols):
    grouped = df.groupby(col)[target].agg(["mean", "count"])
    grouped = grouped.sort_values("mean", ascending=True)
    colors_bar = [
        "#e74c3c" if v > df[target].mean() else "#2ecc71" for v in grouped["mean"]
    ]
    ax.barh(range(len(grouped)), grouped["mean"] * 100, color=colors_bar)
    ax.set_yticks(range(len(grouped)))
    ax.set_yticklabels(grouped.index, fontsize=8)
    ax.axvline(x=df[target].mean() * 100, color="black", linestyle="--", linewidth=0.8)
    ax.set_title(f"Tasa Mora30 por {col}", fontsize=10)
    ax.set_xlabel("Tasa de morosidad (%)")

plt.tight_layout()
plt.savefig("../output/figures/02_categorical_mora_rate.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 02_categorical_mora_rate.png")

# 3d. Análisis de variables continuas
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
continuous_cols = [
    "Ingresos",
    "GastosFamiliares",
    "Edad",
    "TiempoClienteMeses",
    "TiempoActividadAnios",
    "PORCEND",
]

for ax, col in zip(axes.flatten(), continuous_cols):
    df_plot = df[[col, target]].copy()
    df_plot[target] = df_plot[target].map({0: "No Mora", 1: "Mora 30+"})
    ax.boxplot(
        [
            df_plot[df_plot[target] == "No Mora"][col].dropna(),
            df_plot[df_plot[target] == "Mora 30+"][col].dropna(),
        ],
        labels=["No Mora", "Mora 30+"],
    )
    ax.set_title(f"Distribución de {col}", fontsize=10)

plt.tight_layout()
plt.savefig(
    "../output/figures/03_continuous_distributions.png", dpi=150, bbox_inches="tight"
)
plt.close()
print("   -> Guardado: 03_continuous_distributions.png")


[3] Realizando análisis exploratorio...


   -> Guardado: 01_eda_overview.png


   -> Guardado: 02_categorical_mora_rate.png


   -> Guardado: 03_continuous_distributions.png


4. PREPROCESAMIENTO

In [8]:
print("\n[4] Preprocesamiento de datos...")

df_model = df.copy()

# Feature engineering
# Ratio de endeudamiento
income = df_model["Ingresos"].replace(0, df_model["Ingresos"].replace(0, np.nan).median())
df_model["Debt_to_Income"] = df_model["Obligaciones_SistemaFro"] / income
# Ratio gastos/ingresos
df_model["Expense_Ratio"] = df_model["GastosFamiliares"] / income
# Experiencia relativa
df_model["Experience_to_Age"] = df_model["TiempoActividadAnios"] / (
    df_model["Edad"].replace(0, df_model["Edad"].replace(0, np.nan).median())
)
# Antiguedad como cliente
df_model["Cliente_Years"] = df_model["TiempoClienteMeses"] / 12

# Label encoding para categóricas
label_encoders = {}
categorical_features = [
    "OCUPACION",
    "TIPCONTRATO",
    "Estado_Civil",
    "Genero",
    "Nivel_Academico",
    "Tipo_Vivienda",
]

for col in categorical_features:
    le = LabelEncoder()
    df_model[col + "_enc"] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

# Seleccionar features
feature_cols = [
    "ExperienciaSectorFinanciero",
    "PersonasCargo",
    "GastosFamiliares",
    "GastoArriendo",
    "TiempoActividadAnios",
    "Edad",
    "Ingresos",
    "Tipo_Vivienda_enc",
    "TiempoClienteMeses",
    "Tiempo_SistemaFro",
    "PORCEND",
    "Obligaciones_SistemaFro",
    "OCUPACION_enc",
    "TIPCONTRATO_enc",
    "Estado_Civil_enc",
    "Genero_enc",
    "Nivel_Academico_enc",
    "Debt_to_Income",
    "Expense_Ratio",
    "Experience_to_Age",
    "Cliente_Years",
]

# Nota: MoraMax_UltimoSemestre fue excluido por ser variable de fuga de datos (data leakage).
# Esta variable contiene información post-evento sobre la morosidad máxima del cliente,
# lo que permitiría predecir perfectamente Mora30 y Mora60. En un escenario real de
# originación, esta información no estaría disponible al momento de la decisión.

X = df_model[feature_cols]
y = df_model[target]

# Train/test split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Escalar features para Logistic Regression (coeficientes comparables)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"   Train: {len(X_train)} registros ({y_train.mean() * 100:.1f}% mora)")
print(f"   Test:  {len(X_test)} registros ({y_test.mean() * 100:.1f}% mora)")


[4] Preprocesamiento de datos...
   Train: 3236 registros (24.4% mora)
   Test:  1079 registros (24.4% mora)
